# 13_error_analysis_and_case_review

Performs locked-test error analysis for CNN and classical predictions.

In [1]:
from pathlib import Path
import os, json, shutil, zipfile, glob, warnings, math, random
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# 13A. ERROR ANALYSIS AND CASE REVIEW — SETUP


print("RUNNING SCRIPT 13 ERROR ANALYSIS VERSION 2026-06-07")

_candidate_bases = [Path("/content"), Path("/mnt/data"), Path("/tmp")]

def _base_is_usable(p):
    try:
        if not (p.exists() and os.access(p, os.W_OK)):
            return False
        test_project = p / "project_thermography_equine"
        return (not test_project.exists()) or os.access(test_project, os.W_OK)
    except Exception:
        return False

_default_base = next((p for p in _candidate_bases if _base_is_usable(p)), Path("/tmp"))

BASE_DIR = Path(os.environ.get("THERMO_BASE_DIR", str(_default_base)))
PROJECT_NAME = os.environ.get("THERMO_PROJECT_NAME", "project_thermography_equine")
PROJECT_ROOT = BASE_DIR / PROJECT_NAME

DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
METADATA_DIR = DATA_ROOT / "metadata"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"
PROCESSED_DIR = DATA_ROOT / "processed"
CLEAN_IMAGE_DIR = PROCESSED_DIR / "clean_images"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
GRADCAM_DIR = OUTPUT_ROOT / "gradcam"
CASE_REVIEW_DIR = OUTPUT_ROOT / "case_review"

for p in [
    PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR, SPLIT_DATA_DIR, METADATA_DIR,
    ANNOTATIONS_DIR, PROCESSED_DIR, CLEAN_IMAGE_DIR, OUTPUT_ROOT,
    CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR,
    GRADCAM_DIR, CASE_REVIEW_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [PROJECT_ROOT, OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, BASE_DIR, Path("/mnt/data"), Path("/content")]

def existing_roots():
    return [p for p in SEARCH_ROOTS if p.exists()]

def safe_copy(src, dst, overwrite=False):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    try:
        if src.resolve() == dst.resolve():
            return dst
    except Exception:
        pass

    if overwrite or not dst.exists():
        shutil.copy2(src, dst)

    return dst

def find_first(patterns, roots=None):
    roots = roots or existing_roots()

    if isinstance(patterns, str):
        patterns = [patterns]

    candidates = []

    for root in roots:
        if not root.exists():
            continue

        for pattern in patterns:
            candidates.extend(sorted(root.rglob(pattern)))

    candidates = [
        p for p in candidates
        if p.is_file()
        and ".ipynb_checkpoints" not in p.parts
    ]

    if not candidates:
        return None

    return sorted(candidates, key=lambda x: x.stat().st_mtime, reverse=True)[0]

def recover_file(target_name, patterns=None):
    patterns = patterns or [target_name]

    preferred = [
        CONFIG_DIR / target_name,
        REPORTS_DIR / target_name,
        TABLES_DIR / target_name,
        FIGURES_DIR / target_name,
        MODELS_DIR / target_name,
        OUTPUT_ROOT / target_name,
        PROJECT_ROOT / target_name,
        BASE_DIR / target_name,
        Path("/content") / target_name,
        Path("/mnt/data") / target_name,
    ]

    for p in preferred:
        if p.exists() and p.is_file():
            return p

    src = find_first(patterns)

    if src is None:
        return None

    suffix = Path(target_name).suffix.lower()

    if suffix in [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".svg"]:
        dst_primary = FIGURES_DIR / target_name
    elif suffix in [".pt", ".pth", ".joblib", ".pkl"]:
        dst_primary = MODELS_DIR / target_name
    elif target_name.startswith("table_"):
        dst_primary = TABLES_DIR / target_name
    else:
        dst_primary = CONFIG_DIR / target_name

    safe_copy(src, dst_primary)

    if suffix in [".csv", ".json", ".txt"]:
        safe_copy(src, CONFIG_DIR / target_name)
        safe_copy(src, REPORTS_DIR / target_name)

        if target_name.startswith("table_") or "table" in target_name:
            safe_copy(src, TABLES_DIR / target_name)

    return dst_primary

def recover_known_outputs():
    mapping = {
        "classical_features.csv": ["classical_features.csv", "classical_features*.csv"],
        "classical_feature_metadata.json": ["classical_feature_metadata.json"],
        "classical_feature_summary.csv": ["classical_feature_summary.csv"],
        "classical_model_selection_results.csv": ["classical_model_selection_results.csv", "classical_model_selection_results*.csv"],
        "selected_classical_model.json": ["selected_classical_model.json"],
        "selected_classical_model.joblib": ["selected_classical_model.joblib", "selected_classical_model*.joblib"],
        "classical_baseline_metrics.csv": ["classical_baseline_metrics.csv"],
        "table_classical_baseline_metrics.csv": ["table_classical_baseline_metrics.csv"],
        "classical_baseline_predictions.csv": ["classical_baseline_predictions.csv", "classical_baseline_predictions*.csv"],
        "cnn_model_metrics.csv": ["cnn_model_metrics.csv", "cnn_model_metrics*.csv"],
        "table_cnn_model_metrics.csv": ["table_cnn_model_metrics.csv"],
        "cnn_model_predictions.csv": ["cnn_model_predictions.csv", "cnn_model_predictions*.csv"],
        "cnn_model_record.json": ["cnn_model_record.json", "cnn_model_record*.json"],
        "cnn_training_history.csv": ["cnn_training_history.csv"],
        "model_ablation_results.csv": ["model_ablation_results.csv"],
        "model_ablation_summary.csv": ["model_ablation_summary.csv"],
        "table_model_ablation_summary.csv": ["table_model_ablation_summary.csv"],
        "methods_classical_baseline_text.txt": ["methods_classical_baseline_text.txt"],
        "methods_ablation_robustness_text.txt": ["methods_ablation_robustness_text.txt"],
        "fig_classical_baseline_test_roc.png": ["fig_classical_baseline_test_roc.png"],
        "fig_cnn_test_roc.png": ["fig_cnn_test_roc.png"],
        "hotspot_localization_evaluation_diagnostics.json": ["hotspot_localization_evaluation_diagnostics.json"],
        "localization_cohort_definition.json": ["localization_cohort_definition.json"],
    }

    status = {}

    for target, patterns in mapping.items():
        recovered = recover_file(target, patterns)
        status[target] = str(recovered) if recovered else None

    for root in existing_roots():
        for pat in ["cnn_*best*.pt", "cnn_*.pt", "*.pth"]:
            for src in sorted(root.rglob(pat)):
                if src.is_file() and ".ipynb_checkpoints" not in src.parts:
                    safe_copy(src, MODELS_DIR / src.name)

    extracted = []

    for root in existing_roots():
        for z in sorted(root.glob("*.zip")):
            lname = z.name.lower()

            if any(tok in lname for tok in ["image", "clean", "preprocess", "processed", "dataset", "raw", "annotation", "hotspot", "gradcam"]):
                target = PROJECT_ROOT / "_uploaded_zip_extracts" / z.stem
                target.mkdir(parents=True, exist_ok=True)
                marker = target / ".extracted"

                if not marker.exists():
                    with zipfile.ZipFile(z, "r") as zr:
                        zr.extractall(target)
                    marker.write_text(datetime.now(timezone.utc).isoformat(), encoding="utf-8")

                extracted.append(str(target))

    status["extracted_zip_dirs"] = extracted

    return status

recovery_status = recover_known_outputs()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_DIR:", CONFIG_DIR)
print("Recovered known outputs:")

for k, v in recovery_status.items():
    if v:
        print(" -", k, "->", v)

def read_csv_required(path, required_columns=None):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Required file missing: {path}")

    df = pd.read_csv(path)

    if required_columns:
        missing = [c for c in required_columns if c not in df.columns]

        if missing:
            raise KeyError(f"{path.name} is missing required columns: {missing}")

    return df

def standard_metric_col(df):
    df = df.copy()

    if "auc" not in df.columns and "roc_auc" in df.columns:
        df["auc"] = df["roc_auc"]

    if "roc_auc" not in df.columns and "auc" in df.columns:
        df["roc_auc"] = df["auc"]

    if "average_precision" not in df.columns and "pr_auc" in df.columns:
        df["average_precision"] = df["pr_auc"]

    return df

def assert_no_test_selection(record_or_df=None):
    if isinstance(record_or_df, dict):
        if record_or_df.get("test_set_used_for_model_selection") is True:
            raise AssertionError(
                "Test set was used for model selection; this violates the locked analysis protocol."
            )

    return True

RUNNING SCRIPT 13 ERROR ANALYSIS VERSION 2026-06-07
PROJECT_ROOT: /content/project_thermography_equine
CONFIG_DIR: /content/project_thermography_equine/outputs/config
Recovered known outputs:
 - classical_features.csv -> /content/project_thermography_equine/outputs/config/classical_features.csv
 - classical_feature_metadata.json -> /content/project_thermography_equine/outputs/config/classical_feature_metadata.json
 - classical_feature_summary.csv -> /content/project_thermography_equine/outputs/config/classical_feature_summary.csv
 - classical_model_selection_results.csv -> /content/project_thermography_equine/outputs/config/classical_model_selection_results.csv
 - selected_classical_model.json -> /content/project_thermography_equine/outputs/config/selected_classical_model.json
 - selected_classical_model.joblib -> /content/project_thermography_equine/outputs/config/selected_classical_model.joblib
 - classical_baseline_metrics.csv -> /content/project_thermography_equine/outputs/config/c

In [2]:
from sklearn.metrics import confusion_matrix


# 13B. ERROR ANALYSIS AND CASE REVIEW

base = read_csv_required(
    CONFIG_DIR / "classical_baseline_predictions.csv",
    [
        "horse_id",
        "image_name",
        "split",
        "label_binary",
        "classical_probability_pathological",
        "classical_predicted_label_binary",
    ],
)

cnn = read_csv_required(
    CONFIG_DIR / "cnn_model_predictions.csv",
    [
        "horse_id",
        "image_name",
        "split",
        "label_binary",
        "cnn_probability_pathological",
        "cnn_predicted_label_binary",
    ],
)

features_path = CONFIG_DIR / "classical_features.csv"
features = pd.read_csv(features_path) if features_path.exists() else pd.DataFrame()

record_path = CONFIG_DIR / "cnn_model_record.json"

if record_path.exists():
    cnn_record = json.loads(record_path.read_text(encoding="utf-8"))
    assert_no_test_selection(cnn_record)
else:
    cnn_record = {}

test_base = base[base["split"].astype(str).str.lower().eq("test")].copy()
test_cnn = cnn[cnn["split"].astype(str).str.lower().eq("test")].copy()

if test_cnn.empty:
    raise RuntimeError("No CNN test-set rows found. Expected split == 'test'.")

if test_base.empty:
    warnings.warn("No classical baseline test-set rows found. Classical comparison will contain missing values.")

def classify_error(y, pred):
    y = int(y)
    pred = int(pred)

    if y == 1 and pred == 1:
        return "true_positive"

    if y == 0 and pred == 0:
        return "true_negative"

    if y == 0 and pred == 1:
        return "false_positive"

    if y == 1 and pred == 0:
        return "false_negative"

    return "unknown"

review = test_cnn.merge(
    test_base[
        [
            "horse_id",
            "image_name",
            "classical_probability_pathological",
            "classical_predicted_label_binary",
        ]
    ],
    on=["horse_id", "image_name"],
    how="left",
)

review["cnn_error_category"] = [
    classify_error(y, p)
    for y, p in zip(review["label_binary"], review["cnn_predicted_label_binary"])
]

review["classical_error_category"] = [
    classify_error(y, p)
    if pd.notna(p)
    else "classical_prediction_missing"
    for y, p in zip(review["label_binary"], review["classical_predicted_label_binary"])
]

# Merge descriptive feature/QC variables


if not features.empty:
    useful = [
        "horse_id",
        "image_name",
        "img_mean",
        "img_std",
        "top10_mean",
        "top30_mean",
        "entropy_gray",
        "grad_mean",
        "lr_asym_mean_abs",
        "healthy_with_expert_hotspot",
    ]

    useful = [c for c in useful if c in features.columns]

    if {"horse_id", "image_name"}.issubset(useful):
        review = review.merge(
            features[useful],
            on=["horse_id", "image_name"],
            how="left",
            suffixes=("", "_feature"),
        )

# ------------------------------------------------------------
# Rename healthy hotspot flag to avoid biological overclaim
# ------------------------------------------------------------
# Healthy horses should not have inflammatory hotspots by definition.
# If a healthy case has an expert hotspot-like annotation, this is treated
# as a descriptive annotation/QC flag only, not as a biological endpoint.

if "healthy_with_expert_hotspot" in review.columns:
    review = review.rename(
        columns={
            "healthy_with_expert_hotspot": "healthy_with_expert_annotation_flag"
        }
    )

if "healthy_with_expert_hotspot_feature" in review.columns:
    review = review.rename(
        columns={
            "healthy_with_expert_hotspot_feature": "healthy_with_expert_annotation_flag_feature"
        }
    )

annotation_flag_cols = [
    c for c in [
        "healthy_with_expert_annotation_flag",
        "healthy_with_expert_annotation_flag_feature",
    ]
    if c in review.columns
]

for c in annotation_flag_cols:
    review[c] = review[c].fillna(False).astype(bool)

review["case_review_note"] = ""

if annotation_flag_cols:
    any_flag = np.logical_or.reduce([review[c].values for c in annotation_flag_cols])

    review.loc[
        any_flag,
        "case_review_note"
    ] = (
        "Healthy-case expert annotation flag retained for descriptive QC review only; "
        "not used as a predictor and not interpreted as inflammatory hotspot localization."
    )


# Error summaries


summary = (
    review
    .groupby(["cnn_error_category", "label_binary"], dropna=False)
    .size()
    .reset_index(name="n_cases")
)

agreement = (
    review.assign(
        model_agreement=np.where(
            review["cnn_predicted_label_binary"] == review["classical_predicted_label_binary"],
            "agree",
            "disagree",
        )
    )
    .groupby(
        [
            "model_agreement",
            "cnn_error_category",
            "classical_error_category",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="n_cases")
)

n_fp = int((review["cnn_error_category"] == "false_positive").sum())
n_fn = int((review["cnn_error_category"] == "false_negative").sum())
n_tp = int((review["cnn_error_category"] == "true_positive").sum())
n_tn = int((review["cnn_error_category"] == "true_negative").sum())

n_total = int(len(review))
n_positive = int((review["label_binary"] == 1).sum())
n_negative = int((review["label_binary"] == 0).sum())

accuracy = (n_tp + n_tn) / n_total if n_total else np.nan
sensitivity = n_tp / (n_tp + n_fn) if (n_tp + n_fn) else np.nan
specificity = n_tn / (n_tn + n_fp) if (n_tn + n_fp) else np.nan
ppv = n_tp / (n_tp + n_fp) if (n_tp + n_fp) else np.nan
npv = n_tn / (n_tn + n_fn) if (n_tn + n_fn) else np.nan

error_metrics = pd.DataFrame([
    {
        "model": "cnn",
        "analysis_set": "locked_test_set",
        "n_total": n_total,
        "n_positive": n_positive,
        "n_negative": n_negative,
        "true_positive": n_tp,
        "true_negative": n_tn,
        "false_positive": n_fp,
        "false_negative": n_fn,
        "accuracy": accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "positive_predictive_value": ppv,
        "negative_predictive_value": npv,
    }
])

annotation_qc_summary = pd.DataFrame([
    {
        "n_test_cases": n_total,
        "n_healthy_cases": n_negative,
        "n_pathological_cases": n_positive,
        "n_healthy_cases_with_expert_annotation_flag": int(
            review.loc[review["label_binary"].eq(0), annotation_flag_cols].any(axis=1).sum()
        )
        if annotation_flag_cols else 0,
        "interpretation": (
            "Expert annotation flags in clinically healthy cases are retained only "
            "for descriptive QC/case review and are not interpreted as true inflammatory hotspots."
        ),
    }
])


# Save outputs


review.to_csv(CONFIG_DIR / "error_case_review.csv", index=False)
review.to_csv(REPORTS_DIR / "error_case_review.csv", index=False)
review.to_csv(CASE_REVIEW_DIR / "error_case_review.csv", index=False)

summary.to_csv(CONFIG_DIR / "error_case_summary.csv", index=False)
summary.to_csv(REPORTS_DIR / "error_case_summary.csv", index=False)
summary.to_csv(TABLES_DIR / "table_error_case_summary.csv", index=False)

agreement.to_csv(CONFIG_DIR / "model_error_agreement.csv", index=False)
agreement.to_csv(REPORTS_DIR / "model_error_agreement.csv", index=False)

error_metrics.to_csv(CONFIG_DIR / "error_analysis_metrics.csv", index=False)
error_metrics.to_csv(REPORTS_DIR / "error_analysis_metrics.csv", index=False)
error_metrics.to_csv(TABLES_DIR / "table_error_analysis_metrics.csv", index=False)

annotation_qc_summary.to_csv(CONFIG_DIR / "annotation_qc_summary.csv", index=False)
annotation_qc_summary.to_csv(REPORTS_DIR / "annotation_qc_summary.csv", index=False)
annotation_qc_summary.to_csv(TABLES_DIR / "table_annotation_qc_summary.csv", index=False)

diagnostics = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "analysis": "cnn_error_analysis_locked_test_set",
    "n_test_cases": n_total,
    "n_positive": n_positive,
    "n_negative": n_negative,
    "true_positive": n_tp,
    "true_negative": n_tn,
    "false_positive": n_fp,
    "false_negative": n_fn,
    "accuracy": accuracy,
    "sensitivity": sensitivity,
    "specificity": specificity,
    "positive_predictive_value": ppv,
    "negative_predictive_value": npv,
    "healthy_annotation_flag_policy": (
        "Healthy-case expert annotation flags are retained for descriptive QC/case review only. "
        "They are not used as predictors and are not interpreted as inflammatory hotspots."
    ),
    "test_set_used_for_model_selection": bool(cnn_record.get("test_set_used_for_model_selection", False)),
}

for outdir in [CONFIG_DIR, REPORTS_DIR]:
    (outdir / "error_analysis_diagnostics.json").write_text(
        json.dumps(diagnostics, indent=2),
        encoding="utf-8",
    )

summary_text = f"""Error analysis summary

The CNN was evaluated on the locked held-out test set only.

CNN confusion categories were:
- true positives: {n_tp}
- true negatives: {n_tn}
- false positives: {n_fp}
- false negatives: {n_fn}

Derived test-set performance:
- accuracy: {accuracy:.3f}
- sensitivity: {sensitivity:.3f}
- specificity: {specificity:.3f}
- positive predictive value: {ppv:.3f}
- negative predictive value: {npv:.3f}

The case-review table preserves clinical labels and predictions without relabeling any image.
Healthy cases are retained for the primary binary classification error analysis, because specificity and false-positive review require clinically healthy controls.

Expert annotation flags in clinically healthy cases are retained for descriptive QC/case review only.
They are not used as predictors and are not interpreted as inflammatory hotspot localization.
Primary hotspot localization analysis is handled separately in script 12 and is restricted to pathological cases only.
""".strip()

for outdir in [CONFIG_DIR, REPORTS_DIR]:
    (outdir / "error_analysis_summary.txt").write_text(
        summary_text,
        encoding="utf-8",
    )

methods_text = (
    "Error analysis was performed on the locked held-out test set. CNN predictions "
    "were categorized as true positive, true negative, false positive, or false negative "
    "relative to the clinical binary reference label. Healthy cases were retained in "
    "this analysis because they are required for specificity and false-positive review. "
    "Any expert annotation flags present in clinically healthy cases were retained only "
    "as descriptive QC/case-review variables and were not used as predictors or interpreted "
    "as inflammatory hotspot localization. Hotspot localization metrics were evaluated "
    "separately in script 12 and restricted to pathological cases."
)

for outdir in [CONFIG_DIR, REPORTS_DIR]:
    (outdir / "methods_error_analysis_text.txt").write_text(
        methods_text + "\n",
        encoding="utf-8",
    )

display(summary)
display(error_metrics)
display(annotation_qc_summary)
display(review.head())

print(summary_text)

,cnn_error_category,label_binary,n_cases
0,false_negative,1,3
1,false_positive,0,6
2,true_negative,0,34
3,true_positive,1,10


,model,analysis_set,n_total,n_positive,n_negative,true_positive,true_negative,false_positive,false_negative,accuracy,sensitivity,specificity,positive_predictive_value,negative_predictive_value
0,cnn,locked_test_set,53,13,40,10,34,6,3,0.830189,0.769231,0.85,0.625,0.918919


,n_test_cases,n_healthy_cases,n_pathological_cases,n_healthy_cases_with_expert_annotation_flag,interpretation
0,53,40,13,6,Expert annotation flags in clinically healthy ...


,horse_id,image_name,split,label_clinical,label_binary,healthy_with_expert_annotation_flag,clean_image_path,cnn_probability_pathological,cnn_predicted_label_binary,classical_probability_pathological,...,classical_error_category,img_mean,img_std,top10_mean,top30_mean,entropy_gray,grad_mean,lr_asym_mean_abs,healthy_with_expert_annotation_flag_feature,case_review_note
0,0A0F5,0A0F5.jpg,test,healthy,0,False,/content/project_thermography_equine/_uploaded...,0.001057,0,0.342601,...,false_positive,25.339006,56.154491,171.060608,78.832466,2.913057,4.892192,45.909439,False,
1,1CGFM,1CGFM.jpg,test,healthy,0,False,/content/project_thermography_equine/_uploaded...,0.052607,0,0.204530,...,true_negative,41.079739,50.664135,170.474197,91.394402,5.858007,5.456860,49.421913,False,
2,1RDFC,1RDFC.jpg,test,healthy,0,False,/content/project_thermography_equine/_uploaded...,0.003877,0,0.161864,...,true_negative,21.763632,47.895180,143.905243,21.763632,2.802635,3.895565,42.458385,False,
3,31W0A,31W0A.jpg,test,healthy,0,False,/content/project_thermography_equine/_uploaded...,0.045300,0,0.319038,...,false_positive,59.133827,58.780968,204.202072,122.927788,6.093673,5.861998,46.023476,False,
4,6BGWZ,6BGWZ.jpg,test,healthy,0,False,/content/project_thermography_equine/_uploaded...,0.002814,0,0.261420,...,false_positive,25.683115,53.049351,155.880875,79.656075,2.986748,4.426635,45.049507,False,


Error analysis summary

The CNN was evaluated on the locked held-out test set only.

CNN confusion categories were:
- true positives: 10
- true negatives: 34
- false positives: 6
- false negatives: 3

Derived test-set performance:
- accuracy: 0.830
- sensitivity: 0.769
- specificity: 0.850
- positive predictive value: 0.625
- negative predictive value: 0.919

The case-review table preserves clinical labels and predictions without relabeling any image.
Healthy cases are retained for the primary binary classification error analysis, because specificity and false-positive review require clinically healthy controls.

Expert annotation flags in clinically healthy cases are retained for descriptive QC/case review only.
They are not used as predictors and are not interpreted as inflammatory hotspot localization.
Primary hotspot localization analysis is handled separately in script 12 and is restricted to pathological cases only.


## Completion
This notebook writes standardized outputs to `outputs/config`, `outputs/reports`, `outputs/tables`, and/or `outputs/figures`.